# Week 9 Live Coding
## The donor's portfolio

Four GOTV tactics. Each has a published effect and a cost. Which produces the most votes per dollar?

Four things we will do:
1. Compute cost per vote for each tactic
2. Fix the units on canvassing and phone, and watch \$1,000 turn into \$300
3. Compute total votes produced per \$500K
4. Do the Bayesian update

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below to load the data.

In [ ]:
import pandas as pd
import numpy as np

tactics = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk09_cost_effectiveness/data/gotv_tactics.csv')
tactics

## Part 1: Cost per additional vote

Mail 1,000 people. That costs 1,000 × \$0.60 = \$600. An effect of +0.7 pp means 0.7 of every 100 people, so 7 of the 1,000 vote who would not have. \$600 / 7 ≈ \$86 a vote.

The 1,000 cancels, so do it one person at a time:

**cost per vote = cost per person / effect (as a proportion)** = \$0.60 / 0.007 ≈ \$86

Watch the units! A percentage point is not a percent. The effect goes in as a proportion (+0.7 pp = 0.007), not as 0.7. This is a pet peeve of mine.

Let's run it on all four tactics the way the donor's table is written, and then look hard at what we get.

In [ ]:
# Cost per vote, dividing the donor's cost column by the effect column
tactics['cost_per_vote'] = tactics['cost_per_contact'] / (tactics['effect_pp'] / 100)
tactics[['tactic', 'effect_pp', 'cost_per_contact', 'cost_per_vote']]

**Stop and read the `cost_per_vote` column.** The ranking by effect (canvassing > mail > phone > texts) is not the ranking by cost per vote (texts, then mail, then phone, then canvassing).

The best tactic *per person* is not the best tactic *per dollar*. That is the headline of the week.

But one of those four numbers is wrong, and it is the biggest one.

## Part 2: Match the units

\$20 buys one canvassing **conversation**. But +2.0 pp is the effect per **door knocked**, and most doors do not open. Canvassers reach about **30%** of the doors they knock (Week 5).

Knock 1,000 doors. 300 open, so you pay 300 × \$20 = \$6,000. The effect counts all 1,000 doors: 1,000 × 0.02 = 20 votes. \$6,000 / 20 = **\$300**, not \$1,000. Per door, that is \$20 × 0.30 = \$6.

Phone has the same problem. \$2.00 buys a completed call, but +0.5 pp is per number dialed. Assume 25% of dials reach someone, so a dial costs \$0.50.

The `contact_rate` column holds those numbers. Mail and texts are 1.00: you pay for every piece you send.

In [ ]:
# Knock 1,000 doors, pay 300 x $20 = $6,000, and spread that over
# all 1,000 doors: $6 each. Multiplying by the contact rate does that.
tactics['cost_per_attempt'] = tactics['cost_per_contact'] * tactics['contact_rate']
tactics['cost_per_vote'] = tactics['cost_per_attempt'] / (tactics['effect_pp'] / 100)

tactics[['tactic', 'contact_rate', 'cost_per_contact', 'cost_per_attempt', 'cost_per_vote']]

Canvassing went from \$1,000 a vote to **\$300**, and phone from \$400 to **\$100**. Same tactics, same studies. The top and the bottom of the fraction described different people.

Now read the whole column: texts \$80, mail \$86, phone \$100, canvassing \$300. The biggest effect buys the most expensive vote.

(The other route from the slides lands in the same place: \$20 a conversation over the effect on the people you reached, 2.0 / 0.30 = +6.7 pp, is also \$300.)

## Part 3: Total votes per \$500K

How many additional votes does each tactic produce with a \$500K budget?

In [ ]:
budget = 500000

# How many people can you afford to try?
tactics['attempts'] = (budget / tactics['cost_per_attempt'])

# How many additional votes does that produce?
tactics['additional_votes'] = tactics['attempts'] * (tactics['effect_pp'] / 100)

tactics[['tactic', 'cost_per_attempt', 'attempts', 'effect_pp', 'additional_votes']].round(2)

Texts and mail each produce around 6,000 votes, phone 5,000, canvassing about 1,700. Fixing the units cut canvassing's disadvantage from nearly 12x to about 3.5x. It did not erase it.

**Be suspicious of the mail and texts rows.** 5,833 mail votes assumes all 833,333 mailers work as well as the first one. If the donor's races have 200,000 households worth mailing, that is about four mailers to every house, and the fourth does not do what the first did. Texts have the same problem, at 6.25 million texts.

This table prices the *first* vote, not the *last* one. That is the Lackey & Singer point from the slides.

## Part 4: A new canvassing study arrives

A 2024 field experiment finds +0.8 pp (CI [+0.2, +1.4]) in competitive state legislative races. Your prior, from the older literature, is +2.0 pp (CI [+1.0, +3.0]).

**Bayesian updating** is a weighted average, and the weight on each estimate is its **precision**, 1 / SE². Get each SE from its interval: half the width, divided by 1.96.

In [ ]:
# SE from a 95% interval: half the width, divided by 1.96
prior_se = (3.0 - 1.0) / 2 / 1.96
new_se = (1.4 - 0.2) / 2 / 1.96

# Precision is 1 / SE^2. A tight estimate gets a big weight.
prior_precision = 1 / prior_se**2
new_precision = 1 / new_se**2

# The weighted average
updated = ((prior_precision * 2.0 + new_precision * 0.8)
           / (prior_precision + new_precision))

# Precisions add, so the updated SE is 1 / sqrt(total precision)
updated_se = 1 / np.sqrt(prior_precision + new_precision)

print('updated belief:', round(updated, 1))
print('updated SE:', round(updated_se, 2))

**+1.1 pp.** The single new study gets 74% of the weight and dozens of older experiments get 26%, because its interval is so much narrower. The updated SE, 0.26, is smaller than either one alone (0.51 and 0.31).

---

## What you've seen today

- **Cost per vote = cost / effect.** Check that both describe the same people before you divide. Canvassing was off by 3.3x.
- **The ranking by effect is not the ranking by cost per vote.**
- **Bayesian updating is a weighted average, and the weights are precisions.**

**Lying-with-data tag #8:** Ranking tactics by effect size or cost per attempt instead of cost per vote.

Next, open `wk09_problem_set.ipynb`.